In [2]:

import pandas as pd
from pathlib import Path

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:

from pathlib import Path
import pandas as pd

# Move one level up from the notebooks folder
project_root = Path.cwd().parent

file_path = project_root / "data" / "raw" / "olist" / "olist_orders_dataset.csv"

print("Looking for file:", file_path)
print("File exists:", file_path.exists())

orders = pd.read_csv(file_path)

print("Orders dataset loaded successfully!")


Looking for file: e:\Projects\retail-analytics-portfolio\data\raw\olist\olist_orders_dataset.csv
File exists: True
Orders dataset loaded successfully!


In [4]:
    
print("First 5 rows:")
display(orders.head())

print("Dataset dimensions:")
print(orders.shape)

print("Column names:")
print(orders.columns.tolist())


First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Dataset dimensions:
(99441, 8)
Column names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [5]:

print("Data types and non-null counts:")
orders.info()

print("\nMissing values per column:")
print(orders.isna().sum())

print("\nDuplicate rows:")
print(orders.duplicated().sum())


Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783

In [6]:

print("Column names:")
print(orders.columns.tolist())

print("\nData types:")
print(orders.dtypes)


Column names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [7]:

# 1. Check missing values
missing_count = orders.isna().sum()

# 2. Calculate missing-value percentage
missing_percent = (
    orders.isna().mean() * 100
).round(2)

# 3. Check duplicates
duplicate_rows = orders.duplicated().sum()
duplicate_order_ids = orders["order_id"].duplicated().sum()

# 4. Create a summary table
quality_report = pd.DataFrame({
    "Data Type": orders.dtypes.astype(str),
    "Missing Values": missing_count,
    "Missing (%)": missing_percent
})

print("DATA QUALITY REPORT")
display(quality_report)

print("Exact duplicate rows:", duplicate_rows)
print("Repeated order IDs:", duplicate_order_ids)


DATA QUALITY REPORT


,Data Type,Missing Values,Missing (%)
order_id,str,0,0.00
customer_id,str,0,0.00
order_status,str,0,0.00
order_purchase_timestamp,str,0,0.00
order_approved_at,str,160,0.16
order_delivered_carrier_date,str,1783,1.79
order_delivered_customer_date,str,2965,2.98
order_estimated_delivery_date,str,0,0.00


Exact duplicate rows: 0
Repeated order IDs: 0


In [8]:

print("Order status counts:")
display(orders["order_status"].value_counts(dropna=False))

print("Unique order IDs:", orders["order_id"].nunique())
print("Total records:", len(orders))


Order status counts:


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Unique order IDs: 99441
Total records: 99441


In [9]:

# Count missing timestamps for each order status
missing_by_status = (
    orders.groupby("order_status")
    .agg(
        total_orders=("order_id", "size"),
        missing_approval=("order_approved_at",
                          lambda s: s.isna().sum()),
        missing_carrier_date=("order_delivered_carrier_date",
                              lambda s: s.isna().sum()),
        missing_delivery_date=("order_delivered_customer_date",
                               lambda s: s.isna().sum())
    )
    .sort_values("total_orders", ascending=False)
)

display(missing_by_status)


,total_orders,missing_approval,missing_carrier_date,missing_delivery_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [10]:

# Make a separate copy for investigation
orders_check = orders.copy()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Convert strings to datetime values
for column in date_columns:
    orders_check[column] = pd.to_datetime(
        orders_check[column],
        errors="coerce"
    )

print("Updated data types:")
print(orders_check[date_columns].dtypes)

print("\nMissing timestamps after conversion:")
print(orders_check[date_columns].isna().sum())


Updated data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Missing timestamps after conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [11]:

# Check whether approval occurred before purchase
invalid_approval = (
    orders_check["order_approved_at"]
    < orders_check["order_purchase_timestamp"]
).sum()

# Check whether delivery occurred before purchase
invalid_delivery = (
    orders_check["order_delivered_customer_date"]
    < orders_check["order_purchase_timestamp"]
).sum()

print("Approval before purchase:", invalid_approval)
print("Delivery before purchase:", invalid_delivery)


Approval before purchase: 0
Delivery before purchase: 0


In [12]:

# Start with a copy of the dataset containing datetime values
orders_clean = orders_check.copy()

# Calculate delivery duration in days
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Keep the original records and missing timestamps
print("Original rows:", len(orders))
print("Cleaned rows:", len(orders_clean))

print("\nDelivery duration summary:")
display(orders_clean["delivery_days"].describe())

print("\nMissing delivery duration:",
      orders_clean["delivery_days"].isna().sum())


Original rows: 99441
Cleaned rows: 99441

Delivery duration summary:


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_days, dtype: float64


Missing delivery duration: 2965


In [13]:

# Validate the cleaned dataset
assert orders_clean["order_id"].is_unique
assert orders_clean["order_purchase_timestamp"].notna().all()

print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Total records:", len(orders_clean))
print("Negative delivery durations:",
      (orders_clean["delivery_days"] < 0).sum())

# Save to the processed-data folder
output_path = (
    project_root / "data" / "processed" / "orders_clean.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

orders_clean.to_csv(output_path, index=False)

print("\nSaved cleaned dataset to:")
print(output_path)
print("File exists:", output_path.exists())


Unique order IDs: 99441
Total records: 99441
Negative delivery durations: 0

Saved cleaned dataset to:
e:\Projects\retail-analytics-portfolio\data\processed\orders_clean.csv
File exists: True


In [14]:

longest_deliveries = (
    orders_clean[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
    .sort_values("delivery_days", ascending=False)
    .head(10)
)

display(longest_deliveries)


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,delivery_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,209.628611
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,208.351759
61610,440d0d17af552815d15a9e41abe49359,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,195.634016
70307,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,2017-03-08 18:09:02,2017-09-19 14:33:17,194.850174
89130,285ab9426d6982034523a855f55a885e,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,194.633611
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,194.049583
11399,47b40429ed8cce3aee9199792275433f,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,191.463542
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,189.863160
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,188.134618
68769,c27815f7e3dd0b926b58552628481575,delivered,2017-03-15 23:23:17,2017-09-19 17:14:25,187.743843


In [15]:

orders_clean["delivery_delay_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Analyze only orders with actual delivery dates
delivered_orders = orders_clean[
    orders_clean["order_status"].eq("delivered")
].copy()

delivered_orders["delivered_late"] = (
    delivered_orders["delivery_delay_days"] > 0
)

print("Delivered orders:", len(delivered_orders))

print(
    "Average delivery delay (days):",
    round(delivered_orders["delivery_delay_days"].mean(), 2)
)

print(
    "Orders delivered late:",
    int(delivered_orders["delivered_late"].sum())
)

print(
    "Late delivery rate (%):",
    round(delivered_orders["delivered_late"].mean() * 100, 2)
)


Delivered orders: 96478
Average delivery delay (days): -11.18
Orders delivered late: 7826
Late delivery rate (%): 8.11


In [16]:

# Inspect the longest deliveries against estimated dates
display(
    orders_clean[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days"
        ]
    ]
    .sort_values("delivery_days", ascending=False)
    .head(10)
)

# Delivery duration percentiles
print("Delivery duration percentiles:")
display(
    orders_clean["delivery_days"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
)


,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.628611
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.351759
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.634016
70307,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.850174
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.633611
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.049583
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.463542
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.863160
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.134618
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.743843


Delivery duration percentiles:


0.50    10.217755
0.75    15.720327
0.90    23.097604
0.95    29.276016
0.99    46.049913
Name: delivery_days, dtype: float64